In [ ]:
# Procesamiento de Flujos y Orquestación de Pipelines

## PARTE 1: STRUCTURED STREAMING

### 1. Celda 1: Verificar Estructura de Directorios

In [1]:
import os

print("=" * 60)
print("VERIFICACIÓN DE DIRECTORIOS DE STREAMING")
print("=" * 60)

paths = {
    "Input": os.path.expanduser("~/projects/streaming/input"),
    "Checkpoint": os.path.expanduser("~/projects/streaming/checkpoint"),
    "Delta Output": os.path.expanduser("~/projects/streaming/delta_output")
}

for name, path in paths.items():
    exists = os.path.exists(path)
    status = "OK" if exists else "FAIL"
    print(f" {status} {name}: {path}")

# Listar archivos iniciales
input_files = os.listdir(paths["Input"]) if os.path.exists(paths["Input"]) else []
print(f"\nArchivos en input/: {input_files}")

VERIFICACIÓN DE DIRECTORIOS DE STREAMING
 OK Input: /home/david/projects/streaming/input
 OK Checkpoint: /home/david/projects/streaming/checkpoint
 OK Delta Output: /home/david/projects/streaming/delta_output

Archivos en input/: ['ventas_001.csv']


## 2. Celda 2: Crear SparkSession con Streaming

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

print("=" * 60)
print("CREACIÓN DE SPARKSESSION PARA STREAMING")
print("=" * 60)

spark = (SparkSession.builder
    .appName("StructuredStreaming")
    .master("local[*]")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1500m")
    .config("spark.sql.streaming.checkpointLocation", os.path.expanduser("~/projects/streaming/checkpoint"))
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate())

print("SparkSession creada")
print(f" App: {spark.sparkContext.appName}")
print(f" Checkpoint por defecto: ~/projects/streaming/checkpoint")

CREACIÓN DE SPARKSESSION PARA STREAMING


26/06/20 09:27:34 WARN Utils: Your hostname, david-VMware-Virtual-Platform resolves to a loopback address: 127.0.1.1; using 172.16.179.130 instead (on interface ens33)
26/06/20 09:27:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 09:27:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/20 09:27:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession creada
 App: StructuredStreaming
 Checkpoint por defecto: ~/projects/streaming/checkpoint


## 3. Celda 3: Definir Esquema y Configurar File Stream

In [4]:
print("=" * 60)
print("CONFIGURACIÓN DEL FILE STREAM")
print("=" * 60)

# Esquema del archivo CSV de ventas
schema = StructType([
    StructField("id_venta", StringType(), False),
    StructField("id_cliente", StringType(), False),
    StructField("producto", StringType(), False),
    StructField("cantidad", IntegerType(), False),
    StructField("precio_unitario", DoubleType(), False),
    StructField("fecha", DateType(), True)
])

input_path = os.path.expanduser("~/projects/streaming/input")
checkpoint_path = os.path.expanduser("~/projects/streaming/checkpoint")
output_path = os.path.expanduser("~/projects/streaming/delta_output")

print(f"Input: {input_path}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Output: {output_path}")

# Configurar lectura de stream
df_stream = (spark.readStream
    .schema(schema)
    .option("header", "true")
    .option("maxFilesPerTrigger", 1) # Procesar 1 archivo por trigger
    .csv(input_path))

print("Stream configurado")
print("Esquema:")
df_stream.printSchema()

CONFIGURACIÓN DEL FILE STREAM
Input: /home/david/projects/streaming/input
Checkpoint: /home/david/projects/streaming/checkpoint
Output: /home/david/projects/streaming/delta_output
Stream configurado
Esquema:
root
 |-- id_venta: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- fecha: date (nullable = true)



## 4. Celda 4: Escribir Stream a Delta Lake

In [5]:
from pyspark.sql.functions import col, current_timestamp, lit

print("=" * 60)
print("ESCRITURA DEL STREAM A DELTA LAKE")
print("=" * 60)

# Agregar metadatos de procesamiento
df_enriched = df_stream.withColumn("fecha_procesamiento", current_timestamp()).withColumn("batch_id", lit("batch_streaming"))

# Configurar escritura
query = (df_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="10 seconds") # Trigger cada 10 segundos
    .start(output_path))

print("Stream iniciado")
print(f"Query name: {query.name}")
print(f"Status: {query.status}")
print(f"ID: {query.id}")

# Esperar unos segundos para procesar archivos iniciales
import time
time.sleep(5)

ESCRITURA DEL STREAM A DELTA LAKE


26/06/20 09:32:50 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Stream iniciado
Query name: None
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
ID: 4df3c090-e3f2-4c1b-ab15-9dafa24f6ea5


## 5. Celda 5: Verificar Datos Procesados

In [6]:
print("=" * 60)
print("VERIFICACIÓN DE DATOS PROCESADOS")
print("=" * 60)

# Leer datos procesados
df_delta = spark.read.format("delta").load(output_path)

print(f"Registros procesados: {df_delta.count()}")
print("\nDatos:")
df_delta.show(truncate=False)

print("\nEstadísticas:")
df_delta.groupBy("fecha").count().orderBy("fecha").show()

VERIFICACIÓN DE DATOS PROCESADOS


26/06/20 09:34:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Registros procesados: 3

Datos:


+--------+----------+----------------+--------+---------------+----------+-----------------------+---------------+
|id_venta|id_cliente|producto        |cantidad|precio_unitario|fecha     |fecha_procesamiento    |batch_id       |
+--------+----------+----------------+--------+---------------+----------+-----------------------+---------------+
|V001    |C001      |Laptop Pro      |1       |2500.0         |2024-05-01|2026-06-20 09:32:50.999|batch_streaming|
|V002    |C002      |Mouse Gamer     |2       |45.5           |2024-05-01|2026-06-20 09:32:50.999|batch_streaming|
|V003    |C001      |Teclado Mecánico|1       |120.0          |2024-05-02|2026-06-20 09:32:50.999|batch_streaming|
+--------+----------+----------------+--------+---------------+----------+-----------------------+---------------+


Estadísticas:


+----------+-----+
|     fecha|count|
+----------+-----+
|2024-05-01|    2|
|2024-05-02|    1|
+----------+-----+



## 6. Celda 6: Simular Nuevos Datos (Micro-batch)

In [7]:
import time
print("=" * 60)
print("SIMULACIÓN DE NUEVOS DATOS")
print("=" * 60)

# Crear nuevo archivo CSV en input/
nuevas_ventas = """id_venta,id_cliente,producto,cantidad,precio_unitario,fecha
V004,C003,Monitor 4K,2,450.00,2024-05-03
V005,C002,Webcam HD,1,80.00,2024-05-03
V006,C004,Audífonos BT,3,60.00,2024-05-04"""
nuevo_archivo = os.path.expanduser("~/projects/streaming/input/ventas_002.csv")

with open(nuevo_archivo, 'w') as f:
    f.write(nuevas_ventas)

print(f"Archivo creado: {nuevo_archivo}")
print("Esperando procesamiento (15 segundos)...")
time.sleep(15)

# Verificar datos actualizados
df_delta_actualizado = spark.read.format("delta").load(output_path)
print(f"\nRegistros totales después del micro-batch: {df_delta_actualizado.count()}")
df_delta_actualizado.orderBy("fecha").show(truncate=False)

SIMULACIÓN DE NUEVOS DATOS
Archivo creado: /home/david/projects/streaming/input/ventas_002.csv
Esperando procesamiento (15 segundos)...



Registros totales después del micro-batch: 6


+--------+----------+----------------+--------+---------------+----------+-----------------------+---------------+
|id_venta|id_cliente|producto        |cantidad|precio_unitario|fecha     |fecha_procesamiento    |batch_id       |
+--------+----------+----------------+--------+---------------+----------+-----------------------+---------------+
|V001    |C001      |Laptop Pro      |1       |2500.0         |2024-05-01|2026-06-20 09:32:50.999|batch_streaming|
|V002    |C002      |Mouse Gamer     |2       |45.5           |2024-05-01|2026-06-20 09:32:50.999|batch_streaming|
|V003    |C001      |Teclado Mecánico|1       |120.0          |2024-05-02|2026-06-20 09:32:50.999|batch_streaming|
|V004    |C003      |Monitor 4K      |2       |450.0          |2024-05-03|2026-06-20 09:36:00.047|batch_streaming|
|V005    |C002      |Webcam HD       |1       |80.0           |2024-05-03|2026-06-20 09:36:00.047|batch_streaming|
|V006    |C004      |Audífonos BT    |3       |60.0           |2024-05-04|2026-0

## 7. Celda 7: Verificar Checkpoint y Recuperación

In [8]:
print("=" * 60)
print("VERIFICACIÓN DE CHECKPOINTS")
print("=" * 60)

# Listar contenido del directorio checkpoint
import subprocess
result = subprocess.run(["ls", "-la", checkpoint_path], capture_output=True,
text=True)
print("Contenido de checkpoint/:")
print(result.stdout)

# Detener y reiniciar el stream (simulación de fallo y recuperación)
print("\nDeteniendo stream...")
query.stop()
print("Reiniciando stream desde checkpoint...")
query2 = (df_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="10 seconds")
    .start(output_path))
time.sleep(5)

# Verificar que no hay duplicados (exactly-once semantics)
df_final = spark.read.format("delta").load(output_path)
print(f"\nRegistros finales (sin duplicados): {df_final.count()}")
print("Recuperación desde checkpoint exitosa - Exactly-once garantizado")
query2.stop()

VERIFICACIÓN DE CHECKPOINTS
Contenido de checkpoint/:
total 28
drwxrwxr-x 5 david david 4096 Jun 20 09:32 .
drwxrwxr-x 5 david david 4096 Jun 20 09:19 ..
drwxr-xr-x 2 david david 4096 Jun 20 09:36 commits
-rw-r--r-- 1 david david   45 Jun 20 09:32 metadata
-rw-r--r-- 1 david david   12 Jun 20 09:32 .metadata.crc
drwxr-xr-x 2 david david 4096 Jun 20 09:36 offsets
drwxr-xr-x 3 david david 4096 Jun 20 09:32 sources


Deteniendo stream...
Reiniciando stream desde checkpoint...


26/06/20 09:36:57 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
[Stage 38:=============================================>          (41 + 2) / 50]


Registros finales (sin duplicados): 6
Recuperación desde checkpoint exitosa - Exactly-once garantizado


## 